### Bringing in document group data to make reprint group id metadata

In [1]:
import pandas as pd

In [ ]:
# ---------- CONFIG ----------
INPUT_PATH = "reprints_matrix.csv"
OUTPUT_PATH = "reprints_long.csv"

# 1. Load data
df = pd.read_csv(INPUT_PATH)

# 2. Normalize blanks to NaN in the key columns
cols = ["article_id_parent", "article_id_direct", "article_id_truncated", "article_id_paraphrase"]
for col in cols:
    if col in df.columns:
        df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

# 3. Forward-fill the parent so every row knows its original
df["group_parent"] = df["article_id_parent"].ffill()

# If there are any leading rows without a parent, you can drop them:
df = df[df["group_parent"].notna()].copy()

# Helper: base group_id_type (original + _reprint)
df["group_id_type"] = df["group_parent"].astype(str) + "_reprint"

# 4. Build separate DataFrames for original, direct, truncated

# a) Originals (just the parent rows, once each)
orig = (
    df.loc[df["article_id_parent"].notna(), ["article_id_parent", "group_id_type"]]
      .rename(columns={"article_id_parent": "article_id"})
)
orig["reprint_type"] = "original"

# b) Direct reprints
direct = (
    df.loc[df["article_id_direct"].notna(), ["article_id_direct", "group_id_type"]]
      .rename(columns={"article_id_direct": "article_id"})
)
direct["reprint_type"] = "direct"

# c) Truncated reprints
trunc = (
    df.loc[df["article_id_truncated"].notna(), ["article_id_truncated", "group_id_type"]]
      .rename(columns={"article_id_truncated": "article_id"})
)
trunc["reprint_type"] = "truncated"

# 5. Combine them; paraphrase column is ignored by design
out = pd.concat([orig, direct, trunc], ignore_index=True)

# 6. Drop duplicates if same article_id appears multiple times for any reason
out = out.drop_duplicates(subset=["article_id", "group_id_type", "reprint_type"])

# 7. Save final long-form table
out.to_csv(OUTPUT_PATH, index=False)

print(f"Saved reprint mapping to {OUTPUT_PATH}")


Saved grouped reprints to reprints_grouped.csv
